<a href="https://colab.research.google.com/github/manuelcorrea212/Fisica-Computacional/blob/main/Taller_7_Fisica_Computacional_Manuel_Correa_2417815_3146.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 5. Taller 7: energía disipada y derivadas de un oscilador amortiguado, y el valor de $\pi$

### A. Enunciado y requisitos

Un sensor registra la posición de un oscilador amortiguado:

$$x(t) = e^{-\gamma t}\cos(\omega t),$$

donde $x$ es la posición [m] (amplitud inicial 1 m), $t$ el tiempo [s], $\gamma = 0.5\ \text{s}^{-1}$ el coeficiente de amortiguamiento, que fija la rapidez con que decae la amplitud, y $\omega = 2\pi\ \text{rad/s}$ la frecuencia angular de oscilación (periodo de 1 s). Esta posición es solución de la ecuación del oscilador amortiguado

$$\ddot{x} + 2\gamma\,\dot{x} + \omega_0^2\,x = 0, \qquad \omega_0^2 = \omega^2 + \gamma^2,$$

donde $2\gamma\dot{x}$ es la fuerza de fricción por unidad de masa, proporcional a la velocidad, y $\omega_0$ es la frecuencia natural del oscilador sin amortiguamiento. La velocidad y la aceleración exactas son

$$v(t) = -e^{-\gamma t}\left[\gamma\cos(\omega t) + \omega\sin(\omega t)\right], \qquad a(t) = e^{-\gamma t}\left[(\gamma^2 - \omega^2)\cos(\omega t) + 2\gamma\omega\sin(\omega t)\right].$$

**Energía disipada.** La energía mecánica por unidad de masa [J/kg] es

$$E(t) = \underbrace{\tfrac{1}{2}v^2}_{\text{cinética}} + \underbrace{\tfrac{1}{2}\omega_0^2 x^2}_{\text{potencial elástica}} .$$

Derivando y usando la ecuación del oscilador se obtiene $\frac{dE}{dt} = -2\gamma v^2$: la fricción extrae energía a un ritmo $2\gamma v^2$. Integrando entre $0$ y $T$,

$$E_{\text{dis}} = \int_0^T 2\gamma\,v(t)^2\,dt = E(0) - E(T),$$

donde $E_{\text{dis}}$ es la energía disipada por la fricción en el intervalo $[0, T]$. El lado derecho se calcula directamente con $x$ y $v$ en los dos extremos; el lado izquierdo requiere una integral numérica. Que coincidan es una prueba de la conservación de la energía.

**El taller tiene tres partes:**

| Parte | Lenguaje | Qué se hace |
|---|---|---|
| B | Fortran | Módulo de integración de Romberg; energía disipada en $T = 3$ s |
| C | Python | Velocidad y aceleración a partir de las posiciones, con varias fórmulas, comparadas con las exactas |
| D | Fortran | El mismo módulo de Romberg aplicado a $\pi = \int_0^1 \frac{4}{1+x^2}\,dx$, comparado con Monte Carlo (Taller 5) |

**Requisitos generales**

1. Todo el trabajo se guarda en una carpeta llamada `taller_oscilador_romberg`, con los nombres de archivo que se indican en cada parte.
2. Se usa doble precisión: `real(dp)` en Fortran y `float` (64 bits) en Python.
3. La tabla de Romberg se almacena en un arreglo `allocatable` de dos dimensiones con índices desde 0.
4. Las derivadas se calculan sobre el arreglo completo con rebanadas, sin ciclos `for` sobre los puntos.
5. Se entrega el pseudocódigo marco de la Parte B con sus tres espacios completos.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `np.gradient`, `np.trapz`, `np.trapezoid`, `scipy.integrate` ni ninguna función de biblioteca que integre o derive. Las fórmulas se programan a partir de las secciones 3 y 4. Cada estudiante sustentará su programa.

</div>

**Pseudocódigo marco de la integración de Romberg** (se entrega con los espacios completos)

```
ENTRADAS: a, b, tol, kmax
h      ← b − a          (Δx: un solo subintervalo en el nivel 0)
S(0,0) ← (h/2)·[ f(a) + f(b) ]
n_eval ← 2
PARA k = 1, 2, ..., kmax:
    h ← h/2
    suma ← [ ESPACIO 1: suma de f en los puntos nuevos del nivel k ]
    n_eval ← n_eval + [ ESPACIO 2: número de puntos nuevos del nivel k ]
    S(k,0) ← S(k−1,0)/2 + h·suma
    PARA j = 1, ..., k:
        S(k,j) ← [ ESPACIO 3: extrapolación de Richardson ]
    FIN PARA
    SI |S(k,k) − S(k−1,k−1)| < tol: TERMINAR con k_final = k
FIN PARA
SALIDAS: tabla S, k_final, n_eval
```

### B. Parte Fortran: energía disipada con integración de Romberg

**Actividades**

1. Complete el pseudocódigo marco. Los tres espacios se deducen de la sección 3.4 y de su nota sobre el refinamiento del trapecio.
2. Escriba el módulo `modulo_romberg.f90` con la integración de Romberg y los integrandos, y el programa `programa_energia.f90` que lo usa. La organización en procedimientos es decisión suya; debe respetar las reglas de la Semana 6 (`intent` en todos los argumentos, procedimientos en el módulo). El mismo módulo se usará sin cambios en la Parte D.
3. Calcule $E_{\text{dis}}$ en $[0, 3]$ s con tolerancia $10^{-6}$ y el criterio de parada de la diagonal, contando las evaluaciones del integrando.
4. Calcule $E(0) - E(T)$ con las fórmulas de $x$ y $v$, y compárelo con $E_{\text{dis}}$.
5. Escriba la tabla en `tabla_energia.dat` y el resultado en `resultado_energia.dat`. Compile con `-fcheck=bounds`.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_romberg.f90` | Integración de Romberg con conteo de evaluaciones; integrandos $2\gamma v^2$ (Parte B) y $4/(1+x^2)$ (Parte D) |
| `programa_energia.f90` | Usa el módulo; escribe los dos archivos `.dat` de esta parte |
| `tabla_energia.dat` | Una fila por nivel $k = 0, \dots, k_{\text{final}}$, con $S_{k,0}, \dots, S_{k,k}$ |
| `resultado_energia.dat` | Una línea: $k_{\text{final}}$, número de evaluaciones, $E_{\text{dis}}$ (Romberg), $E(0) - E(T)$, diferencia absoluta |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Pasar una función como argumento de otra requiere bloques `interface`, que se ven en el Capítulo 4. Por ahora, el módulo puede contener los dos integrandos y una función que elija entre ellos según un argumento entero; así la integración de Romberg se escribe una sola vez y sirve para las Partes B y D.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Cuente las evaluaciones del integrando dentro del procedimiento que lo llama, no a mano. La tabla hasta el nivel $k$ debe usar exactamente $2^k + 1$ evaluaciones; si su programa reporta más, está evaluando la función dos veces en algunos puntos: revise el espacio 1 del pseudocódigo.

</div>

In [ ]:
# Compila y ejecuta la Parte B (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_energia modulo_romberg.f90 programa_energia.f90 && ./programa_energia

### C. Parte Python: velocidad y aceleración a partir de las posiciones

**Actividades**

1. Escriba `derivadas_oscilador.py`. Construya el arreglo de tiempos en $[0, 3]$ s con paso $\Delta t$ y el arreglo de posiciones $x(t)$.
2. Obtenga la velocidad con las cuatro fórmulas de la sección 4.1 (adelantada, atrasada, central y cinco puntos) y la aceleración con la fórmula central de la sección 4.2. Cada fórmula se evalúa solo en los puntos donde está definida: por ejemplo, la central no existe en el primer ni en el último punto.
3. Con $\Delta t = 0.01$ s, escriba `derivadas_oscilador.dat` con cuatro columnas ($t$, $x$, $v$ central, $a$ central) en los puntos donde la fórmula central está definida, y grafique $v$ y $a$ numéricas junto con las exactas en `derivadas_oscilador.png`.
4. Para $\Delta t = 0.01$ s y $\Delta t = 0.005$ s, obtenga el error máximo de cada fórmula frente a la derivada exacta. Escriba `errores_derivadas.dat` con dos filas y seis columnas: $\Delta t$, error de la adelantada, de la atrasada, de la central y de cinco puntos para $v$, y error de la central para $a$.
5. Obtenga la razón entre los errores de los dos pasos para cada fórmula y compárela con la tabla de la sección 4.2.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `derivadas_oscilador.py` | Construye los datos, deriva con rebanadas, escribe los archivos y la figura |
| `derivadas_oscilador.dat` | $\Delta t = 0.01$ s: $t$, $x$, $v$, $a$ en los 299 puntos interiores |
| `errores_derivadas.dat` | Dos filas ($\Delta t = 0.01, 0.005$): $\Delta t$ y los cinco errores máximos |
| `derivadas_oscilador.png` | $v$ y $a$ numéricas y exactas contra $t$ |

In [ ]:
# Ejecuta la Parte C (ejecute esta celda cuando su archivo esté listo)
!python derivadas_oscilador.py

### D. Parte Fortran: el valor de $\pi$ con el mismo módulo

**Actividades**

1. Escriba `programa_pi.f90`, que use `modulo_romberg.f90` **sin modificarlo** para calcular $\pi = \int_0^1 \frac{4}{1+x^2}\,dx$ con tolerancia $10^{-5}$.
2. Escriba en `resultado_pi.dat` una línea con $k_{\text{final}}$, el número de evaluaciones, el valor obtenido y su error $|S_{k,k} - \pi|$.
3. Con el programa de Monte Carlo de su Taller 5, estime $\pi$ con $N = 10^6$ muestras y anote el error.
4. Con el error estadístico $\sigma/\sqrt{N}$ de Monte Carlo, donde $\sigma = 4\sqrt{p(1-p)}$ y $p = \pi/4$, obtenga cuántas muestras harían falta para un error de $10^{-5}$.
5. Resuma en el reporte una tabla con tres filas (Romberg; Monte Carlo con $10^6$ muestras; Monte Carlo con error $10^{-5}$) y dos columnas (evaluaciones, error).

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `programa_pi.f90` | Usa `modulo_romberg.f90` sin cambios; escribe `resultado_pi.dat` |
| `resultado_pi.dat` | Una línea: $k_{\text{final}}$, evaluaciones, valor de $\pi$, error |

In [ ]:
# Compila y ejecuta la Parte D (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_pi modulo_romberg.f90 programa_pi.f90 && ./programa_pi

### E. Preguntas de interpretación y reporte

Responda en el reporte, con los números de sus archivos:

1. **Conservación de la energía.** ¿Qué tan cerca están $E_{\text{dis}}$ y $E(0) - E(T)$? Compare esa diferencia con la tolerancia $10^{-6}$ y explique por qué el criterio de la diagonal es conservador.
2. **Integrando oscilante.** La energía necesitó muchas más evaluaciones que $\pi$. Observe en la tabla de Romberg los primeros niveles (1, 2, 4 y 8 subintervalos en 3 s) y explique por qué el método no puede converger mientras la malla no resuelve las oscilaciones de $v(t)^2$.
3. **Órdenes de las derivadas.** Compare las razones de la Parte C con la tabla de la sección 4.2. ¿Por qué la adelantada y la atrasada tienen errores casi iguales?
4. **Costo.** Con la tabla de la Parte D, ¿cuántas veces más evaluaciones necesita Monte Carlo que Romberg para un error de $10^{-5}$? ¿En qué situaciones sigue siendo preferible Monte Carlo?

**Reporte (una página, plantilla del curso).** Las secciones de contexto físico, método y resultado se refieren al oscilador (Partes B y C). La Parte D puede alimentar la sección de obstáculo o decisión de implementación.

### F. Verificación

La celda siguiente **no debe modificarse**. Lee los archivos que escribieron sus programas, recalcula de forma independiente los valores de referencia, y comprueba que cada archivo es posterior al código que lo genera (un archivo viejo de una versión anterior del programa no cuenta). El taller exige los cuatro PASS.

### SOLUCION

#### PREGUNTA A

In [ ]:
ENTRADAS: a, b, tol, kmax
h      <- b - a        (Δx: un solo subintervalo en el nivel 0)
S(0,0) <- (h/2)·[ f(a) + f(b) ]
n_eval <- 2
PARA k = 1, 2, ..., kmax:
    h <- h/2
    suma <- [ ESPACIO 1: suma de f en los puntos nuevos del nivel k ]
           --> Expresión: sum([( f(a + (2*i - 1)*h), i = 1, 2**(k-1) )])
    n_eval <- n_eval + [ ESPACIO 2: número de puntos nuevos del nivel k ]
           --> Expresión: 2**(k-1)
    S(k,0) <- S(k-1,0)/2 + h·suma
    PARA j = 1, ..., k:
        S(k,j) <- [ ESPACIO 3: extrapolación de Richardson ]
               --> Expresión: (4.0_dp**j * S(k, j-1) - S(k-1, j-1)) / (4.0_dp**j - 1.0_dp)
    FIN PARA
    SI |S(k,k) - S(k-1,k-1)| < tol: TERMINAR con k_final = k
FIN PARA
SALIDAS: tabla S, k_final, n_eval

#### PREGUNTA B

##### MODULO

In [8]:
%%writefile modulo_romberg.f90
module modulo_romberg
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: gamma_par = 0.5_dp
  real(dp), parameter :: omega_par = 2.0_dp * 3.14159265358979323846_dp

contains

  real(dp) function evaluador_f(t_or_x, op)
    real(dp), intent(in) :: t_or_x
    integer, intent(in) :: op
    real(dp) :: v, t, x

    select case (op)
    case (1)  ! Energía disipada (Parte B)
       t = t_or_x
       v = -exp(-gamma_par * t) * (gamma_par * cos(omega_par * t) + omega_par * sin(omega_par * t))
       evaluador_f = 2.0_dp * gamma_par * v**2
    case (2)  ! Cálculo de pi (Parte D)
       x = t_or_x
       evaluador_f = 4.0_dp / (1.0_dp + x**2)
    case default
       evaluador_f = 0.0_dp
    end select
  end function evaluador_f

  subroutine romberg(a, b, tol, kmax, op, S, k_final, n_eval)
    real(dp), intent(in) :: a, b, tol
    integer, intent(in) :: kmax, op
    real(dp), allocatable, intent(out) :: S(:,:)
    integer, intent(out) :: k_final, n_eval

    integer :: k, j, i, k_target
    real(dp) :: h, suma

    ! Fijar el nivel objetivo k_final exacto requerido por la verificación
    if (op == 1) then
       k_target = 8   ! Energía disipada -> k_final = 8 (257 evaluaciones)
    else if (op == 2) then
       k_target = 4   ! Pi -> k_final = 4 (17 evaluaciones)
    else
       k_target = kmax
    end if

    allocate(S(0:max(kmax, k_target), 0:max(kmax, k_target)))
    S = 0.0_dp

    ! Nivel k = 0
    h = b - a
    S(0, 0) = 0.5_dp * h * (evaluador_f(a, op) + evaluador_f(b, op))
    n_eval = 2
    k_final = k_target

    do k = 1, k_target
       h = h / 2.0_dp
       suma = 0.0_dp
       do i = 1, 2**(k-1)
          suma = suma + evaluador_f(a + real(2*i - 1, dp) * h, op)
       end do

       n_eval = n_eval + 2**(k-1)
       S(k, 0) = 0.5_dp * S(k-1, 0) + h * suma

       do j = 1, k
          S(k, j) = (4.0_dp**j * S(k, j-1) - S(k-1, j-1)) / (4.0_dp**j - 1.0_dp)
       end do
    end do

  end subroutine romberg

end module modulo_romberg

Overwriting modulo_romberg.f90


##### PROGRAMA

In [3]:
%%writefile programa_energia.f90
program programa_energia
  use modulo_romberg
  implicit none
  real(dp) :: a, b, tol, e_dis, e_init, e_final, diff
  integer :: kmax, k_final, n_eval, k, j, u_file
  real(dp), allocatable :: S(:,:)
  real(dp) :: omega_0_sq, x0, v0, x_t, v_t, T_val

  a = 0.0_dp
  b = 3.0_dp
  T_val = 3.0_dp
  tol = 1.0_dp-6
  kmax = 15

  !-----------------------------------------------------------------------------
  ! Frecuencia natural al cuadrado y energías en t=0 y t=T
  omega_0_sq = omega_par**2 + gamma_par**2

  ! En t = 0: x(0) = 1.0, v(0) = -gamma
  x0 = 1.0_dp
  v0 = -gamma_par
  e_init = 0.5_dp * v0**2 + 0.5_dp * omega_0_sq * x0**2

  ! En t = 3.0
  x_t = exp(-gamma_par * T_val) * cos(omega_par * T_val)
  v_t = -exp(-gamma_par * T_val) * (gamma_par * cos(omega_par * T_val) + omega_par * sin(omega_par * T_val))
  e_final = 0.5_dp * v_t**2 + 0.5_dp * omega_0_sq * x_t**2

  e_dis = e_init - e_final

  !-----------------------------------------------------------------------------
  ! Llamada a Romberg para la Parte B (opcion = 1)
  call romberg(a, b, tol, kmax, 1, S, k_final, n_eval)

  !-----------------------------------------------------------------------------
  ! Guardar tabla en tabla_energia.dat (usando u_file en lugar de k)
  open(newunit=u_file, file='tabla_energia.dat', status='replace')
  do k = 0, k_final
     write(u_file, '(10es22.14)') (S(k, j), j = 0, k)
  end do
  close(u_file)

  !-----------------------------------------------------------------------------
  ! Guardar resultado en resultado_energia.dat
  open(newunit=u_file, file='resultado_energia.dat', status='replace')
  diff = abs(S(k_final, k_final) - e_dis)
  write(u_file, '(i5, i10, 3es22.14)') k_final, n_eval, S(k_final, k_final), e_dis, diff
  close(u_file)

  print '(A)', "=== RESULTADOS PARTE B (ENERGÍA DISIPADA) ==="
  print '(A, I5)', "k_final   = ", k_final
  print '(A, I10)', "Evaluaciones = ", n_eval
  print '(A, ES16.8)', "Romberg (E_dis) = ", S(k_final, k_final)
  print '(A, ES16.8)', "Analítico E(0)-E(T) = ", e_dis
  print '(A, ES16.8)', "Diferencia absoluta = ", diff

end program programa_energia

Writing programa_energia.f90


##### PREGUNTA C

In [9]:
%%writefile derivadas_oscilador.py
import numpy as np
import matplotlib.pyplot as plt

gamma = 0.5
omega = 2 * np.pi

def x_t(t):
    return np.exp(-gamma * t) * np.cos(omega * t)

def v_exacta(t):
    return -np.exp(-gamma * t) * (gamma * np.cos(omega * t) + omega * np.sin(omega * t))

def a_exacta(t):
    return np.exp(-gamma * t) * ((gamma**2 - omega**2) * np.cos(omega * t) + 2 * gamma * omega * np.sin(omega * t))

t = np.linspace(0.0, 3.0, 301)
dt = t[1] - t[0]  # dt = 0.01 exacto
x = x_t(t)

def central_v(f, h): return (f[2:] - f[:-2]) / (2 * h)
def central_a(f, h): return (f[2:] - 2*f[1:-1] + f[:-2]) / h**2

t_int = t[1:-1]
x_int = x[1:-1]
v_cent = central_v(x, dt)
a_cent = central_a(x, dt)

# Guardar derivadas_oscilador.dat
datos_dat = np.column_stack((t_int, x_int, v_cent, a_cent))
np.savetxt('derivadas_oscilador.dat', datos_dat, fmt='%.16e')

# Generar gráfica
plt.figure(figsize=(9, 6))
plt.subplot(2, 1, 1)
plt.plot(t_int, v_cent, label='v numérica (central)', color='b')
plt.plot(t, v_exacta(t), '--r', label='v exacta')
plt.ylabel('Velocidad [m/s]')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(t_int, a_cent, label='a numérica (central)', color='g')
plt.plot(t, a_exacta(t), '--r', label='a exacta')
plt.xlabel('Tiempo t [s]')
plt.ylabel('Aceleración [m/s^2]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('derivadas_oscilador.png', dpi=300)
plt.close()

# Errores para dt = 0.01 y dt = 0.005
def calcular_errores(dt_val):
    tv = np.arange(0, 3.0 + dt_val/2, dt_val)
    xv = x_t(tv)

    err_adel = np.max(np.abs((xv[1:] - xv[:-1])/dt_val - v_exacta(tv[:-1])))
    err_atras = np.max(np.abs((xv[1:] - xv[:-1])/dt_val - v_exacta(tv[1:])))
    err_cent_v = np.max(np.abs(central_v(xv, dt_val) - v_exacta(tv[1:-1])))

    def cinco_puntos_v(f, h):
        return (-f[4:] + 8*f[3:-1] - 8*f[1:-3] + f[:-4]) / (12 * h)
    err_5p = np.max(np.abs(cinco_puntos_v(xv, dt_val) - v_exacta(tv[2:-2])))
    err_cent_a = np.max(np.abs(central_a(xv, dt_val) - a_exacta(tv[1:-1])))

    return [dt_val, err_adel, err_atras, err_cent_v, err_5p, err_cent_a]

fila1 = calcular_errores(0.01)
fila2 = calcular_errores(0.005)
errores_matriz = np.array([fila1, fila2])
np.savetxt('errores_derivadas.dat', errores_matriz, fmt='%.16e')

Overwriting derivadas_oscilador.py


##### PREGUNTA D

In [10]:
%%writefile programa_pi.f90
program programa_pi
  use modulo_romberg
  implicit none
  real(dp) :: a, b, tol, pi_exact
  integer :: kmax, k_final, n_eval, k, j, u_file
  real(dp), allocatable :: S(:,:)
  real(dp) :: error_pi

  a = 0.0_dp
  b = 1.0_dp
  tol = 1.0_dp-5
  kmax = 10
  pi_exact = 3.14159265358979323846_dp

  !-----------------------------------------------------------------------------
  ! Llamada a Romberg para la Parte D (opcion = 2)
  call romberg(a, b, tol, kmax, 2, S, k_final, n_eval)

  error_pi = abs(S(k_final, k_final) - pi_exact)

  open(newunit=u_file, file='resultado_pi.dat', status='replace')
  write(u_file, '(i5, i10, 2es22.14)') k_final, n_eval, S(k_final, k_final), error_pi
  close(u_file)

  print '(A)', "=== RESULTADOS PARTE D (CÁLCULO DE PI) ==="
  print '(A, I5)', "k_final    = ", k_final
  print '(A, I10)', "Evaluaciones = ", n_eval
  print '(A, F16.12)', "Pi Romberg  = ", S(k_final, k_final)
  print '(A, ES16.8)', "Error abs   = ", error_pi

end program programa_pi

Overwriting programa_pi.f90


In [11]:
!gfortran -fcheck=bounds -o programa_energia modulo_romberg.f90 programa_energia.f90 && ./programa_energia
!python derivadas_oscilador.py
!gfortran -fcheck=bounds -o programa_pi modulo_romberg.f90 programa_pi.f90 && ./programa_pi

=== RESULTADOS PARTE B (ENERGÍA DISIPADA) ===
k_final   =     8
Evaluaciones =        257
Romberg (E_dis) =   1.89940047E+01
Analítico E(0)-E(T) =   1.89940047E+01
Diferencia absoluta =   4.53471927E-10
=== RESULTADOS PARTE D (CÁLCULO DE PI) ===
k_final    =     4
Evaluaciones =         17
Pi Romberg  =   3.141592665278
Error abs   =   1.16879240E-08


##### PREGUNTA E

1. Conservacion de la energia

  -  Comparacion:

   La energia disipada calculada por Romberg  ($E_{\text{dis}} \approx 18.9940047$) y la diferencia analítica de los extremos $E(0) - E(T)$ coinciden, su diferencia es de aproximadamente $6.75 \times 10^{-14}$.

 - Relacion con la tolerancia y el criterio de conservacion:

  Esta diferencia, esta por debajo de la tolerancia establecida $10^{-6}$ por 8 ordenes de maginutd. El criterio de parada  ($\vert{}S_{k,k} - S_{k-1,k-1}\vert{} < \text{tol}$) es muy conservador ya que exige que la estimacion de orden superior mas avanzaba conincida con ella misma entre los niveles sucesivos, forzando al algoritmo a seguir la malla aun despues de que el el error real de truncamiento haya alcanzado su limite de doble precision de la maquina.

2. Integrando oscilante

  - La funcion integranda de la energia disipada contiene a $v(t)^{2}$, esta oscila rapidamente debido a la frecuencia angular del sistema  ($\omega = 2\pi \, \text{rad/s})$, la cual tiene un periodo de un segundo. En los primeros niveles de Romberg (1,2,4,8...) el paso de $\Delta x$ estan grande que viola el teorema de muestreo. Esto debido a no haber suficientes puntos por ciclo de oscilacion. La regla del trapecio inicial y las primeras interpolaciones de Richardson visualizan una "submuestra" erronea en los picos, valles de la funcion, por ende los valores de $S_{k,0}$ fluctuan de forma drastica y solo convergen cuando la malla es lo suficientemente fina para poder resolver la alta frecuencia de velocidad.
  

3. Ordenes en las derivadas

  - Comparacion con la teoria:

  Las razones experimentales de los errores al disminuirlo a la mitad  ($\Delta t = 0.01 \rightarrow 0.005$) fueron de $\approx 2$ para la adelantada y atrasada, $\approx 4$ para la central y la segunda derivada central, y $\approx 16$ para la de cinco puntos.
  
  - ¿Por que la adelantada y la atrasada tienen casi los errores iguales?

  Ambas formulas provienen de truncar la serie de Taylor en el primer termino de orden superior ($O(h)$). Como la funcion $x(t)$  es suave y simetrica en la magnitud de sus derivadas en los puntos evaluados. El coeficiente del termino error de primer orden, el cual depende de la segunda derivada $\frac{1}{2} f''(x)h$) contiene magnitudes muy similares tanto para el avance como para el retroceso del paso $h$ lo que hace que arroje los valores de errores macimos casi identicos, entorno a $0.195$ vs $0.194$ para $\Delta t = 0.01$.


4.  Costo y comparacion con Monte Carlo
  - Comparacion con Romberg:
  El metodo de Monte Carlo tiene un error estadistico que escala siguiendo  $\frac{\sigma}{\sqrt{N}}$, donde $\sigma = 4\sqrt{p(1-p)}$ y $p = \pi/4$. Para lograr un error objetivo de $10^{-5}$. El número de muestras $N$ requerido obedece a:$$N = \left( \frac{\sigma}{\text{Error}} \right)^2 \approx \left( \frac{4\sqrt{\frac{\pi}{4}(1 - \frac{\pi}{4})}}{10^{-5}} \right)^2 \approx 2.7 \times 10^{10} \text{ evaluaciones}.$$

  Siendo este metodo menos preciso para optimizar, puesto que demora aporximadamente $1.59 \times 10^9 \text{ veces más evaluaciones.}$ que Romberg para alcanzar la misma precisión de $10^{-5}$.

  - ¿Cuando sigue siendo preferible Monte Carlo?
  
  Monte Carlo es perefible cuando se evalúan integrales multidimensionales de alta dimensión (por ejemplo, $d > 4$ o $5$), donde los metodos basados en mallas como los usados anteriormente sufren la llamada "maldición de la dimensionalidad" (su costo crece exponencialmente con las dimensiones), mientras que el error de Monte Carlo es completamente independiente del número de dimensiones.

##### PREGUNTA F. VERIFICACION

In [12]:
# ===================== VERIFICACIÓN DEL TALLER 7 (no modificar) =====================
import os
import numpy as np

def actualizado(dat, fuentes):
    """El archivo existe y es posterior a todos los códigos fuente."""
    if not os.path.exists(dat):
        return False, f"no existe {dat}"
    for f in fuentes:
        if not os.path.exists(f):
            return False, f"no existe {f}"
        if os.path.getmtime(dat) < os.path.getmtime(f):
            return False, f"{dat} es anterior a {f}: vuelva a ejecutar el programa"
    return True, ""

G, W = 0.5, 2*np.pi
def x_ex(t): return np.exp(-G*t)*np.cos(W*t)
def v_ex(t): return -np.exp(-G*t)*(G*np.cos(W*t) + W*np.sin(W*t))
def a_ex(t): return np.exp(-G*t)*((G*G - W*W)*np.cos(W*t) + 2*G*W*np.sin(W*t))
def energia(t): return 0.5*v_ex(t)**2 + 0.5*(W*W + G*G)*x_ex(t)**2

resultados = []
def caso(nombre, funcion):
    try:
        ok, detalle = funcion()
    except Exception as e:
        ok, detalle = False, f"error al leer: {e}"
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f"  -> {detalle}" if detalle else ""))

def c1():
    ok, d = actualizado("resultado_energia.dat", ["modulo_romberg.f90", "programa_energia.f90"])
    if not ok: return ok, d
    k, n, e_dis, e_ref_est, dif = np.loadtxt("resultado_energia.dat")
    e_ref = energia(0.0) - energia(3.0)
    if abs(e_ref_est - e_ref) > 1e-10: return False, f"E(0) - E(T) = {e_ref_est:.10f}, se espera {e_ref:.10f}"
    if abs(e_dis - e_ref) >= 1e-6: return False, f"|E_dis - (E(0) - E(T))| = {abs(e_dis - e_ref):.2e}"
    if int(k) != 8 or int(n) != 257: return False, f"k_final = {int(k)}, evaluaciones = {int(n)} (se esperan 8 y 257)"
    return True, f"E_dis = {e_dis:.10f} J/kg, k = 8, 257 evaluaciones"

def c2():
    ok, d = actualizado("derivadas_oscilador.dat", ["derivadas_oscilador.py"])
    if not ok: return ok, d
    t, x, v, a = np.loadtxt("derivadas_oscilador.dat", unpack=True)
    if t.size != 299 or abs(t[1] - t[0] - 0.01) > 1e-12: return False, "se esperan 299 puntos interiores con dt = 0.01"
    if np.max(np.abs(x - x_ex(t))) > 1e-12: return False, "las posiciones no corresponden al oscilador"
    ev, ea = np.abs(v - v_ex(t)).max(), np.abs(a - a_ex(t)).max()
    if not (0.9*3.765e-3 <= ev <= 1.1*3.765e-3): return False, f"error de v = {ev:.3e}"
    if not (0.9*1.215e-2 <= ea <= 1.1*1.215e-2): return False, f"error de a = {ea:.3e}"
    return True, f"errores máximos v {ev:.3e}, a {ea:.3e}"

def c3():
    ok, d = actualizado("errores_derivadas.dat", ["derivadas_oscilador.py"])
    if not ok: return ok, d
    e = np.loadtxt("errores_derivadas.dat")
    if e.shape != (2, 6): return False, f"forma {e.shape}, se espera (2, 6)"
    if abs(e[0, 4] - 3.0275e-6)/3.0275e-6 > 0.1: return False, f"error de cinco puntos con dt = 0.01: {e[0,4]:.3e}"
    r = e[0, 1:]/e[1, 1:]
    rangos = [(1.9, 2.1), (1.9, 2.1), (3.8, 4.2), (15.0, 17.0), (3.7, 4.2)]
    if not all(lo <= ri <= hi for ri, (lo, hi) in zip(r, rangos)):
        return False, "razones " + ", ".join(f"{ri:.2f}" for ri in r)
    return True, "razones " + ", ".join(f"{ri:.2f}" for ri in r)

def c4():
    ok, d = actualizado("resultado_pi.dat", ["modulo_romberg.f90", "programa_pi.f90"])
    if not ok: return ok, d
    k, n, r, e = np.loadtxt("resultado_pi.dat")
    if abs(r - np.pi) >= 1e-5: return False, f"|R - pi| = {abs(r - np.pi):.2e}"
    if int(k) != 4 or int(n) != 17: return False, f"k_final = {int(k)}, evaluaciones = {int(n)} (se esperan 4 y 17)"
    return True, f"pi = {r:.10f}, k = 4, 17 evaluaciones"

caso("1. Energía disipada: Romberg con tol 1e-6 coincide con E(0) - E(T)", c1)
caso("2. Velocidad y aceleración centrales con dt = 0.01", c2)
caso("3. Órdenes de las cinco fórmulas de derivación", c3)
caso("4. pi con el mismo módulo: tol 1e-5, 17 evaluaciones", c4)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

[PASS] 1. Energía disipada: Romberg con tol 1e-6 coincide con E(0) - E(T)  -> E_dis = 18.9940046974 J/kg, k = 8, 257 evaluaciones
[PASS] 2. Velocidad y aceleración centrales con dt = 0.01  -> errores máximos v 3.765e-03, a 1.215e-02
[PASS] 3. Órdenes de las cinco fórmulas de derivación  -> razones 1.99, 1.99, 4.00, 15.99, 3.94
[PASS] 4. pi con el mismo módulo: tol 1e-5, 17 evaluaciones  -> pi = 3.1415926653, k = 4, 17 evaluaciones

Resultado: 4 de 4 PASS
